# Overview

This notebook implements a Retrieval-Augmented Generation (RAG) chatbot for answering questions about Ghana's constitution and selected acts of parliament. It uses LangChain, HuggingFace, OpenAI, and Gradio to:

- **Load and preprocess legal documents** from a local directory.
- **Split documents into manageable text chunks** for efficient retrieval.
- **Generate vector embeddings** for each chunk using HuggingFace models.
- **Store and retrieve relevant chunks** using a Chroma vector database.
- **Use a language model (LLM)** to answer user questions, leveraging retrieved context.
- **Provide an interactive Gradio web interface** for users to chat with the legal assistant and view the supporting context for each answer.

In [ ]:
import os
import glob
from pathlib import Path
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import OpenAIEmbeddings


from dotenv import load_dotenv

In [ ]:
DB_NAME = "law"
KNOWLEDGE_BASE = "knowledge"
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

In [ ]:
def fetch_books():
    print(f"Fetching books from {KNOWLEDGE_BASE}")
    folders = glob.glob(str(Path(KNOWLEDGE_BASE) / "*"))
    books = []
    for folder in folders:
        book = os.path.basename(folder)
        loader = DirectoryLoader(
            folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"}
        )
        folder_docs = loader.load()
        for doc in folder_docs:
            doc.metadata["book"] = book
            books.append(doc)
    return books

In [ ]:
books = fetch_books()
print(f"Fetched {len(books)} documents.")

In [ ]:
def create_chunks(books):
    print(f"Creating chunks from {len(books)} books")
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=200)
    chunks = text_splitter.split_documents(books)
    return chunks

In [ ]:
chunks = create_chunks(books)

In [ ]:
def create_embeddings(chunks):
    print(f"Creating embeddings from {len(chunks)} chunks")
    if os.path.exists(DB_NAME):
        Chroma(persist_directory=DB_NAME, embedding_function=embeddings).delete_collection()

    vectorstore = Chroma.from_documents(
        documents=chunks, embedding=embeddings, persist_directory=DB_NAME
    )

    collection = vectorstore._collection
    count = collection.count()

    sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
    dimensions = len(sample_embedding)
    print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")
    return vectorstore

In [ ]:
create_embeddings(chunks)

In [ ]:
import gradio as gr
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage, convert_to_messages
from langchain_core.documents import Document
from dotenv import load_dotenv




In [ ]:
load_dotenv(override=True)

In [ ]:
MODEL = "gpt-4.1-nano"

RETRIEVAL_K = 5

In [ ]:
vectorstore = Chroma(
    persist_directory=DB_NAME,
    embedding_function=embeddings
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": RETRIEVAL_K}
)


In [ ]:
llm = ChatOpenAI(temperature=0, model_name=MODEL)

In [ ]:
SYSTEM_PROMPT = """
You are a knowledgeable, friendly legal assistant.
You are chatting with a user about the constitution of Ghana and a few acts of parliament.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}"""

In [ ]:
def fetch_context(question: str) -> list[Document]:
    """
    Retrieve relevant context documents for a question.
    """
    return retriever.invoke(question)

In [ ]:
def combined_question(question: str, history: list[dict] | None = None) -> str:
    """
    Combine all the user's messages into a single string.
    """
    history = history or []
    prior = "\n".join(
        m["content"] for m in history if m.get("role") == "user"
    )
    return f"{prior}\n{question}" if prior else question

In [ ]:
def answer_question(
    question: str,
    history: list[dict] | None = None
) -> tuple[str, list[Document]]:
    """
    Answer the given question with RAG.
    Returns:
        - Generated answer
        - Retrieved context documents
    """
  
    history = history or []

    # Combine question with history for better retrieval
    combined = combined_question(question, history)
    # Retrieve documents
    docs = fetch_context(combined)
    context = "\n\n".join(doc.page_content for doc in docs)

    # Build system prompt with context
    system_prompt = SYSTEM_PROMPT.format(context=context)

    # Construct messages
    messages = [SystemMessage(content=system_prompt)]
    messages.extend(convert_to_messages(history))
    messages.append(HumanMessage(content=question))

    # Invoke LLM
    response = llm.invoke(messages)

    return response.content, docs

In [ ]:
def format_context(context):
    result = "<h2 style='color: #ff7800;'>Relevant Context</h2>\n\n"
    for doc in context:
        print(doc.metadata)
        result += f"<span style='color: #ff7800;'>Source: {doc.metadata['source']}</span>\n\n"
        result += doc.page_content + "\n\n"
    return result

In [ ]:
def rag_chat(history):
    last_message = history[-1]["content"]
    prior = history[:-1]
    answer, context = answer_question(last_message, prior)
    history.append({"role": "assistant", "content": answer})
    return history, format_context(context)

In [ ]:
def put_message_in_chatbot(message, history):
        return "", history + [{"role": "user", "content": message}]

In [ ]:
theme = gr.themes.Soft(font=["Inter", "system-ui", "sans-serif"])

with gr.Blocks(title="Legal Assistant", theme=theme) as ui:
    gr.Markdown("# 📖 Ghanaian Legal Aid")

    with gr.Row():
        with gr.Column(scale=1):
            chatbot = gr.Chatbot(
                label="💬 Conversation", height=600, type="messages", show_copy_button=True
            )
            message = gr.Textbox(
                label="Your Question",
                placeholder="Ask anything about Ghana's constitution orthe Acts of Parliament of Ghana...",
                show_label=False,
            )

        with gr.Column(scale=1):
            context_markdown = gr.Markdown(
                label="📚 Retrieved Context",
                value="*Retrieved context will appear here*",
                container=True,
                height=600,
            )

    message.submit(
        put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]
    ).then(rag_chat, inputs=chatbot, outputs=[chatbot, context_markdown])

    ui.launch(inbrowser=False)